In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from google import genai
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# 1. Auto-select an active Gemini Flash model directly from your API key
client = genai.Client()
available = [
    m.name.removeprefix("models/")
    for m in client.models.list()
    if "gemini" in m.name and "generateContent" in (getattr(m, "supported_actions", None) or ["generateContent"])
]
preferred = ["gemini-flash-latest", "gemini-3.8-flash", "gemini-4-hydrogen", "gemini-3.5-flash-lite"]
FAST_MODEL = next((m for m in preferred if m in available), next(m for m in available if "flash" in m or "gemini" in m))
print("Selected live model:", FAST_MODEL)
print("All Gemini models on your key (first 12):", available[:12])

# 2. Run your live v1 Holiday Job Search Agent
def search_seasonal_jobs(city: str, min_hourly_wage: float) -> dict:
    """Finds sample college holiday break jobs in a city above a minimum hourly wage."""
    sample_jobs = [
        {"id": "LOC-101", "title": "Holiday Retail Associate", "employer": "Northstar Outfitters", "city": "Chicago", "wage": 19.50},
        {"id": "LOC-104", "title": "Winter Break STEM Tutor", "employer": "BrightPath Learning", "city": "Chicago", "wage": 26.00},
        {"id": "REM-201", "title": "Remote Online Math Tutor", "employer": "EduBridge Tutoring", "city": "Remote", "wage": 24.00},
    ]
    matches = [
        j for j in sample_jobs
        if j["wage"] >= min_hourly_wage and (city.lower() in j["city"].lower() or j["city"] == "Remote")
    ]
    return {"status": "success", "matches": matches}

holiday_job_agent = Agent(
    name="holiday_job_planner",
    model=FAST_MODEL,
    instruction="You are a helpful College Holiday Job Search Planner. Use search_seasonal_jobs to recommend verified break jobs.",
    tools=[search_seasonal_jobs],
)

runner = InMemoryRunner(agent=holiday_job_agent)
await runner.run_debug(
    "I'm home in Chicago for winter break and need at least $19/hr. What holiday jobs fit me?",
    verbose=True,
)

ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

In [ ]:
from google import genai
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# 1. Print available Gemini models on your API key
client = genai.Client()
gemini_models = [m.name.removeprefix("models/") for m in client.models.list() if "gemini" in m.name]
print("Available Gemini models (sample):", gemini_models[:12])

# 2. Run your live v1 Holiday Job Search Agent
def search_seasonal_jobs(city: str, min_hourly_wage: float) -> dict:
    """Finds sample college holiday break jobs in a city above a minimum hourly wage."""
    sample_jobs = [
        {"id": "LOC-101", "title": "Holiday Retail Associate", "employer": "Northstar Outfitters", "city": "Chicago", "wage": 19.50},
        {"id": "LOC-104", "title": "Winter Break STEM Tutor", "employer": "BrightPath Learning", "city": "Chicago", "wage": 26.00},
        {"id": "REM-201", "title": "Remote Online Math Tutor", "employer": "EduBridge Tutoring", "city": "Remote", "wage": 24.00},
    ]
    matches = [
        j for j in sample_jobs
        if j["wage"] >= min_hourly_wage and (city.lower() in j["city"].lower() or j["city"] == "Remote")
    ]
    return {"status": "success", "matches": matches}

holiday_job_agent = Agent(
    name="holiday_job_planner",
    model="gemini-flash-latest",
    instruction="You are a helpful College Holiday Job Search Planner. Use search_seasonal_jobs to recommend verified break jobs.",
    tools=[search_seasonal_jobs],
)

runner = InMemoryRunner(agent=holiday_job_agent)
await runner.run_debug(
    "I'm home in Chicago for winter break and need at least $19/hr. What holiday jobs fit me?",
    verbose=True,
)

In [ ]:
from google import genai
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

client = genai.Client()
all_models = [m.name.removeprefix("models/") for m in client.models.list() if "gemini" in m.name]
print("Models on your key:", all_models)

skip = ("image", "tts", "audio", "live", "embedding", "transcribe", "veo", "robotics", "computer-use", "2.5-flash")
candidates = [m for m in all_models if not any(s in m for s in skip)]
# Try flash models first
candidates.sort(key=lambda m: (0 if "flash" in m else 1, m))

WORKING_MODEL = None
for m in candidates:
    try:
        r = client.models.generate_content(model=m, contents="Reply OK")
        print(f"WORKING MODEL FOUND: {m} -> {r.text.strip()}")
        WORKING_MODEL = m
        break
    except Exception as e:
        print(f"  skipping {m}: {type(e).__name__}")

if WORKING_MODEL:
    def search_seasonal_jobs(city: str, min_hourly_wage: float) -> dict:
        """Finds sample college holiday break jobs in a city above a minimum hourly wage."""
        sample_jobs = [
            {"id": "LOC-101", "title": "Holiday Retail Associate", "employer": "Northstar Outfitters", "city": "Chicago", "wage": 19.50},
            {"id": "LOC-104", "title": "Winter Break STEM Tutor", "employer": "BrightPath Learning", "city": "Chicago", "wage": 26.00},
            {"id": "REM-201", "title": "Remote Online Math Tutor", "employer": "EduBridge Tutoring", "city": "Remote", "wage": 24.00},
        ]
        matches = [
            j for j in sample_jobs
            if j["wage"] >= min_hourly_wage and (city.lower() in j["city"].lower() or j["city"] == "Remote")
        ]
        return {"status": "success", "matches": matches}

    holiday_job_agent = Agent(
        name="holiday_job_planner",
        model=WORKING_MODEL,
        instruction="You are a helpful College Holiday Job Search Planner. Use search_seasonal_jobs to recommend verified break jobs.",
        tools=[search_seasonal_jobs],
    )
    runner = InMemoryRunner(agent=holiday_job_agent)
    await runner.run_debug(
        "I'm home in Chicago for winter break and need at least $19/hr. What holiday jobs fit me?",
        verbose=True,
    )

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

def search_seasonal_jobs(city: str, min_hourly_wage: float) -> dict:
    """Finds sample college holiday break jobs in a city above a minimum hourly wage."""
    sample_jobs = [
        {"id": "LOC-101", "title": "Holiday Retail Associate", "employer": "Northstar Outfitters", "city": "Chicago", "wage": 19.50},
        {"id": "LOC-104", "title": "Winter Break STEM Tutor", "employer": "BrightPath Learning", "city": "Chicago", "wage": 26.00},
        {"id": "REM-201", "title": "Remote Online Math Tutor", "employer": "EduBridge Tutoring", "city": "Remote", "wage": 24.00},
    ]
    matches = [
        j for j in sample_jobs
        if j["wage"] >= min_hourly_wage and (city.lower() in j["city"].lower() or j["city"] == "Remote")
    ]
    return {"status": "success", "matches": matches}

holiday_job_agent = Agent(
    name="holiday_job_planner",
    model="gemini-3.5-flash",
    instruction="You are a helpful College Holiday Job Search Planner. Use search_seasonal_jobs to recommend verified break jobs.",
    tools=[search_seasonal_jobs],
)

runner = InMemoryRunner(agent=holiday_job_agent)
await runner.run_debug(
    "I'm home in Chicago for winter break and need at least $19/hr. What holiday jobs fit me?",
    verbose=True,
)

In [ ]:
await runner.run_debug(
    "I'm home in Chicago for winter break and need at least $20/hr. What jobs fit me?",
    verbose=True,
)

In [ ]:
await runner.run_debug(
    "Which one of those is fully remote, and why would it fit a short winter break?",
    verbose=True,
)

In [ ]:
await runner.run_debug(
    "Find me a winter break job in Chicago paying at least $50/hr.",
    verbose=True,
)

In [ ]:
print("Holiday Job Search Agent is ready! Type your question below (or 'quit' to stop).\n")

while True:
    user_msg = input("You: ").strip()
    if not user_msg or user_msg.lower() in {"quit", "exit", "q"}:
        print("Chat ended.")
        break
    await runner.run_debug(user_msg, verbose=False)
    